# Observability for Agentic RAG Systems

This guide covers **why observability matters** for agentic RAG pipelines, compares the leading platforms available in 2025-2026, and walks through a practical integration using **Langfuse** with this project.

## 1. Why Observability Matters

Traditional applications fail loudly: an HTTP 500, a stack trace, a timeout. LLM-based systems fail **silently**. An agent can select the wrong tool, retrieve stale documents, hallucinate a correlation, and present the result confidently - all without raising a single exception.

Observability for agentic systems answers questions that standard APM tools cannot:

- **Did the retriever surface the right chunks?** (Context Precision / Recall)
- **Is the generated answer grounded in the retrieved data?** (Faithfulness)
- **Which tool did the agent call, and was it the right choice?** (Tool Call tracing)
- **Where is latency accumulating across the graph?** (Span-level timing)
- **How much does each query cost?** (Token-level cost tracking)

## 2. Key Evaluation Metrics for RAG

Before choosing an observability platform, it helps to understand what you're measuring.

| Dimension | Metric | What it measures | Typical failure cause |
|-----------|--------|------------------|-----------------------|
| **Retrieval** | Context Precision | Ratio of relevant chunks vs. noise | Poor chunking strategy, embedding model mismatch |
| **Retrieval** | Context Recall | Whether *all* needed information was found | Suboptimal search algorithm, missing metadata filters |
| **Generation** | Faithfulness (Groundedness) | Whether the answer sticks to retrieved context | Overly permissive system prompt, model tendency to confabulate |
| **Generation** | Answer Relevance | Alignment between user intent and response | Query misinterpretation, no query rewriting step |

Most platforms compute these via **LLM-as-a-judge** (using a strong model to evaluate outputs). Frameworks like [Ragas](https://docs.ragas.io/) and [DeepEval](https://docs.confident-ai.com/) provide standardized implementations.

## 3. Telemetry Anatomy: Sessions, Traces, and Spans

Agentic systems produce **hierarchical, non-linear execution trees**. The standard telemetry model breaks down into three levels:

- **Session** - A full multi-turn conversation. Tracks memory coherence and behavioral drift over time.
- **Trace** - One complete request-response cycle within a session. Contains timing and cost data.
- **Span** - An atomic unit of work inside a trace: an LLM call, a tool invocation, a retrieval step.

In an agentic graph like this project's, a single user question can produce dozens of spans: query rewriting, parallel sub-agent invocations, multiple search/retrieve cycles, context compression, and final aggregation.

## 4. Platform Comparison

| Platform | Type | Open Source | Best for | Pricing (free tier) |
|----------|------|------------|----------|---------------------|
| **[Langfuse](https://langfuse.com)** | Tracing & prompt management | Yes (MIT core; EE features for enterprise) | Self-hosted deployments, data sovereignty, framework-agnostic setups | Cloud: 50K units/mo — Self-hosted: unlimited |
| **[LangSmith](https://smith.langchain.com)** | Tracing & debugging | No (proprietary) | LangChain/LangGraph native workflows, graph state visualization | 5K traces/mo |
| **[Arize Phoenix](https://phoenix.arize.com)** | MLOps observability | Yes (ELv2 — cannot offer as managed service to third parties) | Embedding drift detection, vector space analysis | Unlimited (self-hosted) |
| **[AgentOps](https://agentops.ai)** | Agent-specific monitoring | Yes (MIT) | Multi-agent collaboration analysis, role-based cost tracking | Free tier available |
| **[Braintrust](https://braintrust.dev)** | E2E eval + CI/CD | Partial (SDK open) | Production-to-eval feedback loops, deployment quality gates | Free tier available |
| **[Helicone](https://helicone.ai)** | AI Gateway / Proxy | Yes (Apache 2.0) | Cost optimization, semantic caching, multi-provider routing | 10K requests/mo |

### Integration approaches

These platforms use three distinct instrumentation strategies:

1. **SDK-based** (Braintrust, LangSmith) — Decorators and wrappers in your code. Deep visibility into internal logic, but tighter coupling.
2. **OpenTelemetry** (Arize Phoenix, Langfuse) — Standard OTEL spans. Interoperable with Datadog/Grafana/Splunk. Future-proof.
3. **Proxy/Gateway** (Helicone) — Network-level interception. Zero code changes, but limited to input/output of LLM calls (no internal agent logic).

### Which one should you use?

- **LangChain/LangGraph stack** → LangSmith gives the deepest integration with zero config. Langfuse is the best open-source alternative.
- **Self-hosted / data sovereignty** → Langfuse (MIT core, unlimited self-hosted) or Arize Phoenix (ELv2, unlimited self-hosted).
- **Multi-agent debugging** → AgentOps specializes in agent collaboration patterns.
- **Cost optimization at scale** → Helicone as a gateway with semantic caching.
- **CI/CD quality gates** → Braintrust for automated regression prevention.


## 5. Practical Integration: Langfuse with this Project

This project uses Langfuse because it's open-source, framework-agnostic, and trivial to integrate with LangGraph via the callback system.

### How it works

LangGraph propagates `callbacks` from the top-level `graph.stream()` config down to every node, subgraph, and LLM call automatically. This means we only need to inject the Langfuse callback handler in **one place** — the `get_config()` method — and every operation gets traced.

```
graph.stream(input, config={"callbacks": [langfuse_handler]})
    └─ summarize_history   → llm.invoke() ← traced
    └─ rewrite_query       → llm.invoke() ← traced
    └─ agent (subgraph, via Send())
    │   └─ orchestrator    → llm.invoke() ← traced
    │   └─ tools           → search/retrieve ← traced
    │   └─ compress_context → llm.invoke() ← traced
    └─ aggregate_answers   → llm.invoke() ← traced
```


### Setup

**Step 1** — Install the dependency:

```bash
pip install langfuse
```

> If you installed the project dependencies via `pip install -r requirements.txt`, langfuse is already included.

**Step 2** — Configure environment variables:

```
LANGFUSE_ENABLED=true
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_BASE_URL=https://cloud.langfuse.com  # or your self-hosted URL
```

Copy `.env.example` to `.env` and fill in the values.

**Step 3** — Run the app normally:

```bash
python project/app.py
```

Traces will appear in your Langfuse dashboard.

To **disable** tracing, set `LANGFUSE_ENABLED=false` in your `.env`. The application runs identically either way.


### The integration code

The full implementation lives in three files. Here's what each one does:

In [ ]:
# core/observability.py — Observability class (simplified view)

from langfuse import get_client
from langfuse.langchain import CallbackHandler

class Observability:
    def __init__(self):
        # Reads LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY, LANGFUSE_BASE_URL
        # from environment variables automatically.
        self._client = get_client()
        self._handler = CallbackHandler()

    def get_handler(self):
        return self._handler

    def flush(self):
        self._client.flush()

# The actual class includes defensive imports (try/except ImportError)
# and graceful fallback when langfuse is not installed or configured.


In [ ]:
# core/rag_system.py

from core.observability import Observability

class RAGSystem:
    def __init__(self):
        self.vector_db = VectorDbManager()
        self.parent_store = ParentStoreManager()
        self.chunker = DocumentChuncker()
        self.observability = Observability()  
        # ...

    def get_config(self):
        cfg = {
            "configurable": {"thread_id": self.thread_id},
            "recursion_limit": self.recursion_limit,
        }
        handler = self.observability.get_handler()
        if handler:
            cfg["callbacks"] = [handler]
        return cfg

# LangGraph propagates callbacks to ALL nodes and subgraphs automatically.

In [ ]:
# core/chat_interface.py — Flush after session is cleared

def chat(self, message, history):
    # Simplified view. The actual implementation also handles the human-in-the-loop
    # case (current_state.next) for query clarification interrupts.
    for chunk, metadata in self.rag_system.agent_graph.stream(
        {"messages": [HumanMessage(content=message.strip())]},
        config=self.rag_system.get_config(),
        stream_mode="messages"
    ):
        yield chunk

def clear_session(self):
    self.rag_system.reset_thread()
    self.rag_system.observability.flush()  # Ensures pending traces are sent


### What gets traced

With this setup, Langfuse captures:

- Every LLM call across all graph nodes (orchestrator, query rewriting, summarization, compression, aggregation)
- Structured output parsing (e.g. `QueryAnalysis` in the rewrite step)
- Tool invocations (`search_child_chunks`, `retrieve_parent_chunks`) with arguments and results
- The full graph execution flow including parallel sub-agent fan-out
- Token counts and latency per span

## 6. Langfuse Hosting Options

### Option A: Langfuse Cloud

Sign up at [cloud.langfuse.com](https://cloud.langfuse.com), create an organization, then create a project and grab your API keys. Set them in `.env`. Free up to 50K units/month (units = traces + observations + scores). Data is stored on Langfuse servers (EU/US).

### Option B: Self-hosted with Docker Compose

For full data sovereignty, clone and run the official Langfuse stack:

```bash
git clone https://github.com/langfuse/langfuse.git
cd langfuse
docker compose up
```

Then open `http://localhost:3000`, create an account and a project, and set the API keys in your `.env`:

```
LANGFUSE_ENABLED=true
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_BASE_URL=http://localhost:3000
```

See the [official self-hosting documentation](https://langfuse.com/self-hosting) for advanced configuration, Kubernetes deployment, and production hardening.


## 7. Further Reading

- [Langfuse documentation](https://langfuse.com/docs)
- [LangGraph streaming](https://langchain-ai.github.io/langgraph/how-tos/streaming/)
- [Ragas evaluation framework](https://docs.ragas.io/)
- [DeepEval — LLM testing](https://docs.confident-ai.com/)
- [OpenTelemetry semantic conventions for GenAI](https://opentelemetry.io/docs/specs/semconv/gen-ai/)
